# Transformers & Attention 


### The story so far

You've already worked with sequential models — RNNs and LSTMs. They process text **one word at a time**, left to right, passing a hidden state forward. They work, but they have two big problems:

1. **The bottleneck problem** — the entire meaning of a long sentence gets compressed into one fixed-size vector. By the time you reach word 50, the model has mostly forgotten word 1.
2. **They can't be parallelized** — word 5 can't be processed until word 4 is done. This makes training slow.

**Transformers solve both problems at once.** Instead of processing words one by one, a transformer looks at all words simultaneously and learns which words should pay attention to which other words.

---

### What you'll build in this notebook

| Part | What | Why |
|---|---|---|
| 1 | Understand attention with a toy example | Build the intuition before the math |
| 2 | Implement scaled dot-product attention from scratch | Understand what the model actually computes |
| 3 | Implement a single attention head from scratch | See queries, keys, values in action |
| 4 | Fine-tune BERT on a sentiment classification task | Learn by doing with a real model |
| 5 | Visualize attention weights | See what the model is actually attending to |
| 6 | Connect to DocForge — how CodeT5 uses transformers | Tie it back to your project |

---
### Ground rules
- Every `# TODO` cell is yours to fill in
- Don't skip the written answer cells — they matter more than the code

---
## Setup
Run this cell as-is.

In [ ]:
!pip install transformers datasets torch -q

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
torch.manual_seed(42)
np.random.seed(42)

print(f'PyTorch: {torch.__version__}')

---
# PART 1 — The Attention Intuition

Before any code, let's build the intuition.

Consider this sentence:

> *"The animal didn't cross the street because **it** was too tired."*

What does **"it"** refer to? The animal, or the street?

Humans instantly know it's the animal — because "tired" applies to animals, not streets. To figure this out, your brain didn't just look at the word before "it" — it looked at the **whole sentence at once** and decided which words were relevant to understanding "it".

That's exactly what attention does. For every word in a sentence, it computes a **score** against every other word — how much should this word "pay attention to" that one? These scores become weights, and the final representation of each word is a weighted combination of all other words.

---
### The three vectors: Query, Key, Value

Attention uses three learned projections of each word:

- **Query (Q):** "What am I looking for?"
- **Key (K):** "What do I contain?"
- **Value (V):** "What information do I pass forward if selected?"

Think of it like a search engine:
- Your **query** is the search term
- Each word's **key** is its metadata tag
- The **value** is the actual content you get back

The attention score between word A and word B = `Q_A · K_B` (dot product)

---
### The formula

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- $QK^T$ — dot product of queries and keys → raw attention scores
- $\sqrt{d_k}$ — scaling factor (prevents softmax from saturating for large dimensions)
- $\text{softmax}$ — converts scores to probabilities that sum to 1
- $\times V$ — weighted sum of values

### 1.1 — Toy attention by hand

**Task:** Before writing any neural network code, let's compute attention manually on a tiny 3-word sentence.

Pretend our sentence is: `["cat", "sat", "mat"]`

We'll represent each word as a 4-dimensional vector (already computed for you).
Your job is to compute the attention output step by step.

In [ ]:
# 3 words, 4-dimensional embeddings — pretend these came from a lookup table
words = ['cat', 'sat', 'mat']
X = torch.tensor([
    [1.0, 0.0, 1.0, 0.0],   # cat
    [0.0, 1.0, 0.0, 1.0],   # sat
    [1.0, 1.0, 0.0, 0.0],   # mat
])  # shape: (3, 4) = (seq_len, d_model)

# For simplicity: Q = K = V = X (we'll learn separate projections in Part 2)
Q = X.clone()
K = X.clone()
V = X.clone()

d_k = Q.shape[-1]  # dimension of keys = 4
print(f'Q shape: {Q.shape} | K shape: {K.shape} | V shape: {V.shape}')
print(f'd_k = {d_k}')

# TODO: Step 1 — Compute raw attention scores
# Formula: scores = Q @ K.T  (matrix multiply Q with transpose of K)
# Result shape should be (3, 3) — score for every word pair
# scores = ???


# TODO: Step 2 — Scale the scores by sqrt(d_k)
# This prevents the dot products from getting too large in high dimensions
# scaled_scores = ???


# TODO: Step 3 — Apply softmax along the last dimension (dim=-1)
# This converts scores to attention weights that sum to 1 per row
# weights = ???


# TODO: Step 4 — Compute the output as weighted sum of values
# output = weights @ V
# output = ???


# TODO: Step 5 — Print the attention weights matrix
# Row i = "how much does word i attend to each other word"
# Print it as a DataFrame with words as index and columns


# TODO: Step 6 — Visualize the attention weights as a heatmap
# x-axis = words being attended TO
# y-axis = words doing the attending
# Use plt.imshow or sns.heatmap
# Title: 'Toy Attention Weights'


In [ ]:
# TODO: Answer these questions as comments

# Q: What shape is the attention weights matrix? What does each row represent?
# Answer:

# Q: Do the weights in each row sum to 1? Why must they?
# Answer:

# Q: Why do we divide by sqrt(d_k) before softmax?
#    Hint: what happens to dot products when d_k is large (e.g. 512)?
# Answer:


---
# PART 2 — Implement Scaled Dot-Product Attention

Now we formalize what you just did into a reusable function.

Real transformers don't use Q=K=V=X directly. Instead, they **learn three separate weight matrices** $W_Q$, $W_K$, $W_V$ and project the input into separate query, key, and value spaces:

$$Q = X W_Q \quad K = X W_K \quad V = X W_V$$

This lets the model learn different "question asking" (Q) and "answering" (K, V) behaviors independently.

We also add an optional **mask** — used in decoders to prevent a position from attending to future positions (you can't look ahead when generating text word by word).

### 2.1 — Implement the attention function

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Compute scaled dot-product attention.

    Args:
        Q: Query matrix  — shape (batch, seq_len, d_k)
        K: Key matrix    — shape (batch, seq_len, d_k)
        V: Value matrix  — shape (batch, seq_len, d_v)
        mask: Optional boolean mask — shape (batch, seq_len, seq_len)
              True = position is masked (set score to -inf before softmax)

    Returns:
        output:  shape (batch, seq_len, d_v)
        weights: shape (batch, seq_len, seq_len)  ← the attention weights
    """
    d_k = Q.shape[-1]

    # TODO: Compute raw scores — Q times K transposed
    # Use torch.matmul(Q, K.transpose(-2, -1))
    # scores shape: (batch, seq_len, seq_len)
    # scores = ???


    # TODO: Scale by sqrt(d_k)
    # scores = ???


    # TODO: Apply mask if provided
    # Where mask is True, set score to float('-inf')
    # Hint: use scores.masked_fill(mask, float('-inf'))


    # TODO: Softmax over last dimension
    # weights = ???


    # TODO: Weighted sum of values
    # output = ???


    return output, weights


# --- Test your function ---
batch, seq_len, d_k, d_v = 2, 5, 8, 8
Q_test = torch.randn(batch, seq_len, d_k)
K_test = torch.randn(batch, seq_len, d_k)
V_test = torch.randn(batch, seq_len, d_v)

out, w = scaled_dot_product_attention(Q_test, K_test, V_test)

# TODO: Print the shapes of out and w
# out should be (2, 5, 8), w should be (2, 5, 5)


# TODO: Verify that attention weights sum to 1 along last dimension
# Print the sum across dim=-1 for the first batch item
# All values should be very close to 1.0


### 2.2 — Test the causal mask (decoder-style)

In a language model decoder, position 3 should NOT be able to attend to positions 4 or 5 — it hasn't seen them yet. This is enforced with a **causal mask** (also called a look-ahead mask).

In [ ]:
# TODO: Create a causal mask for seq_len=5
# A causal mask is an upper triangular matrix of True values
# (True = masked = cannot attend)
# Position i can attend to positions 0..i but NOT i+1..seq_len-1
# Hint: use torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
seq_len = 5
# causal_mask = ???


# TODO: Print the mask — you should see True in the upper triangle


# TODO: Run your attention function with this mask
# Expand mask to batch dimension: causal_mask.unsqueeze(0).expand(batch, -1, -1)


# TODO: Plot the resulting attention weights (first batch item)
# You should see 0s in the upper triangle — positions can't attend to the future


In [ ]:
# TODO: Answer as comments

# Q: Why do decoders need a causal mask but encoders don't?
# Answer:

# Q: What does it mean for a weight to be 0 in the attention matrix?
# Answer:

# Q: CodeT5 has both an encoder and a decoder.
#    Which part would use a causal mask? Which wouldn't?
# Answer:


---
# PART 3 — Implement a Single Attention Head

Now we add the **learned projections** — the weight matrices $W_Q$, $W_K$, $W_V$ that transform the input into query, key, and value spaces.

A single attention head is:
1. Three linear layers: `W_Q`, `W_K`, `W_V` (no bias)
2. Feed input through all three → get Q, K, V
3. Run scaled dot-product attention on Q, K, V
4. One final linear layer `W_O` to project the output back

This is the fundamental building block of every transformer.

### 3.1 — Implement AttentionHead

In [ ]:
class AttentionHead(nn.Module):
    """
    A single attention head.

    Args:
        d_model: Dimension of the input embeddings
        d_head:  Dimension of Q/K/V projections
    """
    def __init__(self, d_model, d_head):
        super().__init__()

        # TODO: Define three linear layers (no bias) for Q, K, V projections
        # Each maps d_model -> d_head
        # self.W_q = ???
        # self.W_k = ???
        # self.W_v = ???


        # TODO: Define one output linear layer (no bias)
        # Maps d_head -> d_model (project back to original dimension)
        # self.W_o = ???


    def forward(self, x, mask=None):
        """
        Args:
            x:    Input tensor — shape (batch, seq_len, d_model)
            mask: Optional causal mask
        Returns:
            output:  shape (batch, seq_len, d_model)
            weights: shape (batch, seq_len, seq_len)
        """
        # TODO: Project x through W_q, W_k, W_v to get Q, K, V
        # Q = ???
        # K = ???
        # V = ???


        # TODO: Call your scaled_dot_product_attention function
        # attn_out, weights = ???


        # TODO: Project through W_o and return
        # output = ???


        return output, weights


# --- Test ---
d_model, d_head = 32, 16
head = AttentionHead(d_model=d_model, d_head=d_head)

x_test = torch.randn(2, 10, d_model)   # batch=2, seq_len=10, d_model=32
out, weights = head(x_test)

# TODO: Print output shape (should be 2, 10, 32) and weights shape (should be 2, 10, 10)


# TODO: Count and print the number of trainable parameters in this head
# Hint: sum(p.numel() for p in head.parameters())


### 3.2 — Why Multi-Head Attention?

Real transformers use **multi-head attention** — they run several attention heads in parallel, each learning to focus on different aspects of the input. One head might learn syntax, another might learn coreference (like "it" → "animal"), another might learn position.

The outputs of all heads are concatenated and projected:
$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W_O$$

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-head attention: run h attention heads in parallel, then concatenate.

    Args:
        d_model:   Dimension of input embeddings
        num_heads: Number of attention heads
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, 'd_model must be divisible by num_heads'

        # TODO: Compute d_head = d_model // num_heads
        # self.d_head = ???


        # TODO: Create a list of num_heads AttentionHead modules
        # Use nn.ModuleList so PyTorch tracks their parameters
        # self.heads = ???


        # TODO: Final output projection: maps (num_heads * d_head) -> d_model
        # self.W_o = ???


    def forward(self, x, mask=None):
        # TODO: Run all heads on x, collect outputs and weights
        # head_outputs = [head(x, mask)[0] for head in self.heads]
        # all_weights  = [head(x, mask)[1] for head in self.heads]
        # Note: running forward twice is inefficient — can you fix it?


        # TODO: Concatenate head outputs along last dimension
        # concat shape: (batch, seq_len, num_heads * d_head)
        # concat = ???


        # TODO: Project through W_o
        # output shape: (batch, seq_len, d_model)
        # output = ???


        return output, all_weights


# --- Test ---
mha = MultiHeadAttention(d_model=32, num_heads=4)
out, all_w = mha(x_test)

# TODO: Print output shape (should be 2, 10, 32)
# TODO: Print how many attention heads there are and each head's d_head
# TODO: Count total trainable parameters in the full multi-head attention module


In [ ]:
# TODO: Answer as comments

# Q: BERT-base uses d_model=768 and num_heads=12. What is d_head?
# Answer:

# Q: Why is it better to have many smaller heads rather than one big head?
#    Think about what different heads could learn independently.
# Answer:

# Q: If you double num_heads but keep d_model the same,
#    does the total parameter count change? Why or why not?
# Answer:


---
# PART 4 — Fine-tune BERT on Sentiment Classification

Theory is good. But the best way to understand transformers is to use one.

We'll fine-tune **BERT-base** on the SST-2 sentiment dataset — movie reviews labelled as positive or negative. This is the same fine-tuning pattern used by CodeT5 for code documentation.

**BERT architecture recap:**
- 12 transformer encoder layers
- 12 attention heads per layer
- d_model = 768
- 110M parameters total
- Pre-trained on masked language modeling (predict hidden words) + next sentence prediction

For classification, we take the `[CLS]` token output (the first token) and pass it through a linear classifier.

### 4.1 — Load dataset and tokenizer

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load SST-2 — binary sentiment (0=negative, 1=positive)
print('Loading SST-2 dataset...')
sst2 = load_dataset('glue', 'sst2')
print(sst2)

# TODO: Print 3 example sentences and their labels from the train split
# Label 0 = negative, label 1 = positive


# TODO: Load the BERT tokenizer
# tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')


# TODO: Tokenize one example sentence and print:
# - the input_ids
# - the decoded tokens (use tokenizer.convert_ids_to_tokens(input_ids))
# Can you spot the [CLS] and [SEP] special tokens?


In [ ]:
# TODO: Answer as comments

# Q: What is the [CLS] token and why is it at the start of every BERT input?
# Answer:

# Q: What is the [SEP] token for?
# Answer:

# Q: What does 'uncased' mean in 'bert-base-uncased'?
# Answer:


### 4.2 — Tokenize the full dataset

In [ ]:
# TODO: Write a tokenize_fn(batch) function that:
# - tokenizes batch['sentence']
# - uses max_length=128, padding='max_length', truncation=True
# - returns the tokenized dict


# TODO: Apply tokenize_fn to sst2 using .map(batched=True)
# Remove the 'sentence' and 'idx' columns after tokenizing
# Set format to 'torch'


# TODO: Print the column names and one sample from the tokenized train set
# You should see: input_ids, attention_mask, token_type_ids, label


### 4.3 — Load BERT and add a classification head

In [ ]:
from transformers import AutoModelForSequenceClassification

# TODO: Load bert-base-uncased for sequence classification with num_labels=2
# model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)


# TODO: Print total parameter count and trainable parameter count


# TODO: Print the model architecture (just print(model))
# Try to identify: the embedding layer, the 12 encoder layers, the classifier head


### 4.4 — Train

In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate as ev

accuracy_metric = ev.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)


# TODO: Create TrainingArguments with:
# output_dir='/kaggle/working/bert_sst2' (change depending on your running environment)
# num_train_epochs=2 (keep it short — BERT trains fast on SST-2)
# per_device_train_batch_size=32
# per_device_eval_batch_size=64
# learning_rate=2e-5
# evaluation_strategy='epoch'
# save_strategy='epoch'
# load_best_model_at_end=True
# metric_for_best_model='accuracy'
# logging_steps=50
# report_to='none'


# TODO: Create Trainer with model, args, train/eval datasets, compute_metrics
# Use tokenized['train'] and tokenized['validation'] as splits


# TODO: Run trainer.train()


# TODO: Print final training loss and best validation accuracy


In [ ]:
# TODO: Answer as comments

# Q: We only trained for 2 epochs. Why does BERT converge so quickly on SST-2?
#    Think about what pre-training already gave it.
# Answer:

# Q: The learning rate is 2e-5 — much smaller than what you used for CodeT5 (5e-5).
#    Why do we use a small learning rate when fine-tuning a large pre-trained model?
# Answer:

# Q: BERT is an encoder-only model. Why can't you use BERT directly for text generation?
# Answer:


---
# PART 5 — Visualize Attention Weights

One of the most powerful things about transformers is that attention weights are **interpretable** — you can actually look at what the model is attending to.

BERT returns attention weights from every layer and every head if you ask for them. Let's visualize them.

### 5.1 — Extract attention weights from BERT

In [ ]:
# Pick a sentence to analyze
sentence = "The cat sat on the mat because it was comfortable"

# TODO: Tokenize the sentence (without padding, truncation=True, return_tensors='pt')
# inputs = tokenizer(sentence, return_tensors='pt')


# TODO: Get the decoded tokens so we can label the heatmap axes
# tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])


# TODO: Run the model with output_attentions=True
# with torch.no_grad():
#     outputs = model(**inputs, output_attentions=True)


# TODO: Print how many layers of attention weights are returned
# and the shape of each — should be (batch, num_heads, seq_len, seq_len)
# outputs.attentions is a tuple of tensors, one per layer


### 5.2 — Plot attention heatmaps

In [ ]:
# TODO: Choose one layer (try layer 0 and layer 11 — first and last)
# and plot the attention weights for all 12 heads as a 3x4 grid of heatmaps
# For each head:
#   - x-axis = tokens being attended TO
#   - y-axis = tokens doing the attending
#   - use sns.heatmap with annot=False, cmap='Blues'
#   - title each subplot 'Head N'

LAYER = 0   # try changing this to 11

# attn = outputs.attentions[LAYER][0]   # shape: (num_heads, seq_len, seq_len)

# TODO: Plot the 3x4 grid


# TODO: Also make a single heatmap of the AVERAGE attention across all heads
# avg_attn = attn.mean(dim=0)  # shape: (seq_len, seq_len)


In [ ]:
# TODO: Answer as comments — look carefully at your heatmaps

# Q: Can you find a head in layer 0 that mostly attends to the [CLS] token?
#    What head number is it? What might this head be learning?
# Answer:

# Q: In the sentence 'The cat sat on the mat because it was comfortable',
#    can you find any head where 'it' has high attention weight toward 'cat'?
#    Which layer and head? (It's okay if you can't find it — explain why it's hard)
# Answer:

# Q: Do early layers (layer 0) look different from late layers (layer 11)?
#    Describe what you observe.
# Answer:


---
# PART 6 — Connection to DocForge: How CodeT5 Uses Transformers

Everything you've built in this notebook is inside CodeT5. Let's map it.

CodeT5 is an **encoder-decoder** transformer:
- The **encoder** reads the source code — full bidirectional attention (like BERT)
- The **decoder** generates the docstring — causal attention (like what you masked in Part 2) + cross-attention to the encoder

**Cross-attention** is the bridge between encoder and decoder:
- Q comes from the decoder ("what am I trying to generate?")
- K and V come from the encoder output ("what did the code say?")
- So at every decoding step, the model asks: which parts of the code are relevant to the word I'm generating right now?

### 6.1 — Inspect CodeT5's architecture

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Load CodeT5-base (same model from your Notebook 3a)
print('Loading CodeT5-base...')
codet5      = AutoModelForSeq2SeqLM.from_pretrained('Salesforce/codet5-base')
codet5_tok  = AutoTokenizer.from_pretrained('Salesforce/codet5-base', use_fast=False)

# TODO: Print total parameter count


# TODO: Find and print the number of encoder layers and decoder layers
# Hint: look at codet5.config
# print(codet5.config)


# TODO: Print d_model (hidden size) and num_heads for CodeT5


In [ ]:
# TODO: Run CodeT5 on a small code snippet and get attention weights
code_snippet = """
def add(a, b):
    return a + b
"""

# TODO: Tokenize the code_snippet as input
# inputs = codet5_tok(code_snippet.strip(), return_tensors='pt')


# TODO: Run the encoder with output_attentions=True
# with torch.no_grad():
#     encoder_outputs = codet5.encoder(**inputs, output_attentions=True)


# TODO: Print the shape of the encoder attention weights
# and the number of layers


# TODO: Plot the average attention heatmap across all heads in layer 0
# Label axes with the actual tokens from the code snippet
# Which tokens attend most to each other?


In [ ]:
# TODO: Answer as comments

# Q: How many attention heads does CodeT5-base have? What is d_head?
# Answer:

# Q: In the encoder attention for your code snippet,
#    which tokens seem to attend strongly to each other?
#    Does this make sense given the code structure?
# Answer:

# Q: The decoder has TWO types of attention: self-attention and cross-attention.
#    In your own words, what is each one doing during docstring generation?
# Answer:

# Q: BERT is encoder-only. CodeT5 is encoder-decoder.
#    Explain in 2-3 sentences why you can't use BERT directly for
#    code documentation generation.
# Answer:


### 6.2 — Generate a docstring with CodeT5

In [ ]:
# TODO: Use CodeT5 to generate a docstring for the code snippet above
# Format the input the same way as Notebook 2: 'Summarize python: {code}'


# TODO: Call codet5.generate() with num_beams=4, max_new_tokens=128


# TODO: Decode the output and print it


# TODO: Now try a slightly more complex function — write your own 5-10 line
# Python function and generate its docstring
# Does CodeT5 describe it correctly?


---
## Done! Final Reflection

Before you finish, write answers to these questions. These are the core ideas from this notebook — if you can answer them clearly, you genuinely understand transformers.

In [ ]:
# FINAL QUESTIONS — answer all of these

# Q1: What is the fundamental problem with RNNs that transformers solve?
#     Give one concrete example where an RNN would struggle.
# Answer:

# Q2: Explain the Query-Key-Value mechanism in your own words.
#     Use an analogy that is NOT a search engine.
# Answer:

# Q3: Why does BERT use bidirectional attention but a language model
#     like Llama uses causal (unidirectional) attention?
# Answer:

# Q4: What is the difference between self-attention and cross-attention?
#     Where does each appear in CodeT5?
# Answer:

# Q5: You fine-tuned BERT on SST-2 and CodeT5 on Code2Doc.
#     In both cases, the pre-trained model already knew something useful.
#     What did BERT already know? What did CodeT5 already know?
# Answer:

# Q6: You implemented attention with O(n^2) memory complexity
#     (the attention matrix is seq_len x seq_len).
#     Why is this a problem for very long sequences?
#     Can you think of a way to approximate attention to reduce this?
# Answer:
